# 05 - Demo Full Pipeline

End-to-end con 10-15 imagenes. Muestra: original, segmentada, paleta, JSON completo y tiempos.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.pipeline import FashionPipeline

In [ ]:
pipeline = FashionPipeline('../config/pipeline_config.yaml')
print('Pipeline inicializado.')

In [ ]:
sample_dir = Path('../data/raw/demo')
images = sorted([p for p in sample_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])[:15]
print(f'Imagenes: {len(images)}')

## Procesar batch y medir tiempos

In [ ]:
t0 = time.time()
results = pipeline.process_batch(images)
t_total = time.time() - t0
print(f'Tiempo total: {t_total:.1f}s | promedio: {t_total/len(images):.2f}s por imagen')

## Visualizar resultados

In [ ]:
def show_result(image_path, result):
    img = Image.open(image_path)
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(img); ax[0].axis('off'); ax[0].set_title(Path(image_path).name)
    ax[1].axis('off')
    summary = []
    if 'tipo' in result:
        summary.append(f"tipo: {result['tipo'].get('label')}")
    if 'estilo' in result:
        summary.append(f"estilo: {result['estilo'].get('label')}")
    if 'color' in result and result['color']:
        summary.append(f"color: {result['color']['dominant'].get('name')}")
    if 'marca' in result:
        summary.append(f"marca: {result['marca'].get('label')}")
    ax[1].text(0.05, 0.5, '\n'.join(summary), fontsize=12, va='center')
    plt.tight_layout(); plt.show()

for p, r in zip(images, results):
    show_result(p, r)

## JSON completo de un ejemplo

In [ ]:
print(json.dumps(results[0], indent=2, ensure_ascii=False))

## Discusion y limitaciones
- Limitaciones observadas en CPU (tiempos de inferencia)
- Casos de error del segmentador
- Color naming en prendas multi-color
- Proximos pasos: datos custom, fine-tuning con mas clases